# Download HumanoidMimicGen initial skills

Downloads **Box Table to Shelf** and **Push Shelf Forward** from the pinned public benchmark dataset. Each task is verified, archived to your Google Drive, and accompanied by a SHA-256 checksum and manifest. Temporary Colab copies are removed after the Drive archive is complete.

You need sufficient free Colab disk for one task at a time and enough Google Drive storage for both archives. These are training demonstrations, not pretrained policy checkpoints. Do not enter AWS credentials into this notebook.


In [ ]:
%pip -q install --upgrade "huggingface_hub[hf_xet]"


In [ ]:
from pathlib import Path
import subprocess

repo = Path("/content/hmg-training")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/athulkrishnaaei/humanoidmimicgen.git", str(repo)], check=True)
print("Training fork:", repo)


## Select skills and download

Run one task first if your Drive space is limited. You can rerun this cell later with the other task. Existing archives are checked by SHA-256 and skipped. Google will ask you to authorize the Drive mount.


In [ ]:
import sys

tasks = [
    "07_box_table_to_shelf",
    "04_push_shelf_forward",
]
subprocess.run(
    [sys.executable, str(repo / "scripts" / "colab_download_initial_skills.py"), "--tasks", *tasks],
    check=True,
)


## Move archives to AWS

The files are saved under `MyDrive/HumanoidMimicGen-initial-skills/`. Transfer them using a storage route approved by your company. On AWS, put both `.tar` files and their `.sha256` files in one directory, then run:

```bash
cd /data/hmg-archives
sha256sum -c 07_box_table_to_shelf.tar.sha256
sha256sum -c 04_push_shelf_forward.tar.sha256
mkdir -p /data/hmg-initial-skills
tar -xf 07_box_table_to_shelf.tar -C /data/hmg-initial-skills
tar -xf 04_push_shelf_forward.tar -C /data/hmg-initial-skills
python3 scripts/verify_initial_skills_data.py /data/hmg-initial-skills \
  --task 07_box_table_to_shelf --task 04_push_shelf_forward
```

Then follow [`INITIAL_SKILLS.md`](https://github.com/athulkrishnaaei/humanoidmimicgen/blob/main/INITIAL_SKILLS.md) in the fork. The AWS training process uses the transferred local dataset and does not need Hugging Face access.
